## Training multi-fidelity neural networks for small/large amplitude oscillatory shear (S/LAOS) rheometry
The output of this notebook is the model's stored training, which is used to predict the oscillatory shear stress.

### Importing the dependencies

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
from time import time
from datetime import datetime
import os
import sys
import math
import tensorflow as tf
import numpy as np
import scipy.optimize
import pandas as pd
import matplotlib.pyplot as plt
from numpy import random
import itertools

# Standard library
import os
import sys
from time import time

# Numerical / scientific computing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.integrate import odeint

# Scikit-learn core utilities
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error
from sklearn.compose import TransformedTargetRegressor

# Scikit-learn models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor, RegressorChain

# Gaussian Processes
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel

# XGBoost
import xgboost as xgb


### Loading the Lo-Fi and Hi-Fi data
We generated the Lo-Fi data using a Maxwell model with inputs being the time, maximum strain, and angular frequency. However, and as stated in the manuscript, the same set of inputs fails to yield accurate stress predictions for MFNN training. Therefore, the inputs here are the strain, strain rate, and angular frequency, with the output being the shear stress.

If the input data is noisy, one can use the `moving_average_filter()` function below to smoothen the data.

In [ ]:
DTYPE='float32'
tf.keras.backend.set_floatx(DTYPE)
SEED=8
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

log10 = tf.experimental.numpy.log10
path = os.getcwd()
files = os.listdir(path)
df = {}
Shuffle = False

url_hf_raw = r"/home/alanh/projects/PunLab/mfnn/data/processed/pig_pstat/combined_pig_pstat.py.xlsx"
url_lf, url_hf = r"/home/alanh/projects/PunLab/mfnn/notebooks/Data_LF_SAOS_pig_pstat.xlsx", r"/home/alanh/projects/PunLab/mfnn/notebooks/Data_HF_SAOS_pig_pstat.xlsx"

df_LF = pd.read_excel(url_lf, sheet_name=None)
data_LF = [
    [k, v] for k, v in df_LF.items() 
]  # NOTE: Adjust to include different sheets #k is the sheet name, v is the pandas df

df_HF = pd.read_excel(url_hf, sheet_name=None)
data_HF = [
    [k, v] for k, v in df_HF.items() 
]  # NOTE: Adjust to include different sheets

# Helper function to concatenate all sheets
def load_and_concat_sheets(data_list):
    frames = []
    for sheet_name, df in data_list:
        # Drop NaNs just like before
        df = df.dropna()
        # Optional: Subsample if data is too large (currently commented out)
        # df = df.iloc[::20] 
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

# 1. Process Lo-Fi Data (All Sheets)
df_LF_concat = load_and_concat_sheets(data_LF)

x1_d_LF = tf.reshape(tf.convert_to_tensor(df_LF_concat['Strain'], dtype=DTYPE),(-1,1))
x2_d_LF = tf.reshape(tf.convert_to_tensor(df_LF_concat['StrainRate'], dtype=DTYPE), (-1,1))
x3_d_LF = tf.reshape(tf.convert_to_tensor(df_LF_concat['AngFreq'], dtype=DTYPE), (-1,1))
x4_d_LF = tf.reshape(tf.convert_to_tensor(df_LF_concat['Time'], dtype=DTYPE), (-1,1))
x5_d_LF = tf.reshape(tf.convert_to_tensor(df_LF_concat['G0'], dtype=DTYPE), (-1,1))
y1_d_LF = tf.reshape(tf.convert_to_tensor(df_LF_concat['Stress'], dtype=DTYPE), (-1,1))

# 2. Process Hi-Fi Data (All Sheets)
df_HF_concat = load_and_concat_sheets(data_HF)

x1_d_HF = tf.reshape(tf.convert_to_tensor(df_HF_concat['Strain'], dtype=DTYPE),(-1,1))
x2_d_HF = tf.reshape(tf.convert_to_tensor(df_HF_concat['StrainRate'], dtype=DTYPE), (-1,1))
x3_d_HF = tf.reshape(tf.convert_to_tensor(df_HF_concat['AngFreq'], dtype=DTYPE), (-1,1))
x4_d_HF = tf.reshape(tf.convert_to_tensor(df_HF_concat['Time'], dtype=DTYPE), (-1,1))
x5_d_HF = tf.reshape(tf.convert_to_tensor(df_HF_concat['G0'], dtype=DTYPE), (-1,1))
y1_d_HF = tf.reshape(tf.convert_to_tensor(df_HF_concat['Stress'], dtype=DTYPE), (-1,1))

# Moving Average Filter (applied to the concatenated HF data)
def moving_average_filter(data, window_size):
    window = np.ones(window_size) / window_size
    smoothed_data = np.convolve(data, window, mode='same')
    return smoothed_data


# Calculate bounds based on the full dataset
x1min, x1max = np.min(x1_d_HF), np.max(x1_d_HF)
x2min, x2max = np.min(x2_d_HF), np.max(x2_d_HF)
x3min, x3max = np.min(x3_d_HF), np.max(x3_d_HF)


# LF: Add x5_d_LF (G0) to inputs
X_data_LF = tf.concat([x1_d_LF, x2_d_LF, x3_d_LF, x5_d_LF], axis=1)
y_data_LF = tf.concat([y1_d_LF], axis=1)  

# HF: Add x5_d_HF (G0) to inputs
X_data_HF = tf.concat([x1_d_HF, x2_d_HF, x3_d_HF, x5_d_HF], axis=1)
y_data_HF = tf.concat([y1_d_HF], axis=1)  

# Update input dimension
in_dim, out_dim = 4, 1

if Shuffle:
    X_data_HF = X_data_HF.numpy()
    X_data_LF = X_data_LF.numpy()
    y_data_HF = y_data_HF.numpy()
    y_data_LF = y_data_LF.numpy()

    # Get the number of samples
    num_samples_HF = X_data_HF.shape[0]
    num_samples_LF = X_data_LF.shape[0]

    # Create shuffled indices for both HF and LF data
    shuffled_indices_HF = np.random.permutation(num_samples_HF)
    shuffled_indices_LF = np.random.permutation(num_samples_LF)

    # Shuffle the data using the shuffled indices
    X_data_HF = X_data_HF[shuffled_indices_HF]
    y_data_HF = y_data_HF[shuffled_indices_HF]

    X_data_LF = X_data_LF[shuffled_indices_LF]
    y_data_LF = y_data_LF[shuffled_indices_LF]

In [ ]:
# Check if LF and HF stress ranges are comparable per sheet
for i in range(min(len(data_LF), len(data_HF))):
    lf = data_LF[i][1]
    hf = data_HF[i][1]
    lf_range = lf['Stress'].max() - lf['Stress'].min()
    hf_range = hf['Stress'].max() - hf['Stress'].min()
    ratio = lf_range / (hf_range + 1e-12)
    flag = " ← CHECK" if not (0.3 < ratio < 3.0) else ""
    print(f"Sheet {i}: LF stress range={lf_range:.4f}, HF={hf_range:.4f}, ratio={ratio:.2f}{flag}")

### Data normalization
In this case, it is important to normalize the data between -1 and 1.

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

# Initialize the scalers
scaler_X = StandardScaler()
scaler_y = StandardScaler()

# Fit the scalers on the Hi-Fi data (to establish the transformation parameters)
X_combined = np.vstack([X_data_HF.numpy(), X_data_LF.numpy()])
y_combined = np.vstack([y_data_HF.numpy(), y_data_LF.numpy()])
scaler_X.fit(X_combined)
scaler_y.fit(y_combined)

def normalize_data(X_data, y_data, scaler_X, scaler_y):
    # Handle both TF tensors and raw Numpy arrays
    X_np = X_data.numpy() if hasattr(X_data, 'numpy') else X_data
    y_np = y_data.numpy() if hasattr(y_data, 'numpy') else y_data
    
    # Transform
    X_norm = scaler_X.transform(X_np)
    y_norm = scaler_y.transform(y_np)
    
    # Convert back to float32 tensors for TensorFlow
    return tf.convert_to_tensor(X_norm, dtype=tf.float32), tf.convert_to_tensor(y_norm, dtype=tf.float32)

print("StandardScaler fitted and data normalized.")

# Apply normalization
X_data_HF, y_data_HF = normalize_data(X_data_HF, y_data_HF, scaler_X, scaler_y)
X_data_LF, y_data_LF = normalize_data(X_data_LF, y_data_LF, scaler_X, scaler_y)

In [ ]:
# After loading and before training:
print("LF y normalized range:", 
      float(tf.reduce_min(y_data_LF)), float(tf.reduce_max(y_data_LF)))
print("HF y normalized range:", 
      float(tf.reduce_min(y_data_HF)), float(tf.reduce_max(y_data_HF)))

# The LF normalized range should be a strict subset of HF range
# If LF extends BEYOND HF in normalized space, you have an extrapolation problem

In [ ]:
# ===== DIAGNOSTIC 4: Visualize LF data per sheet to see if it tracks HF =====
# Run this in Notebook 2 after loading
import matplotlib.pyplot as plt
n = min(5, len(data_LF))
for i in range(n):
    lf = data_LF[i][1]
    hf = data_HF[i][1] if i < len(data_HF) else None
    plt.figure()
    plt.plot(lf['Strain'], lf['Stress'], 'b--', label='LF')
    if hf is not None:
        plt.scatter(hf['Strain'], hf['Stress'], s=2, c='k', label='HF')
    plt.title(f"Sheet {i}")
    plt.legend(); plt.show()

### Defining the NN
Here, `PINN_NeuralNet` is a `tf.keras.Model` instance with several fully-connected layers.

In [ ]:
# Define model architecture
class PINN_NeuralNet(tf.keras.Model):
    """ Set basic architecture of the PINN model."""

    def get_config(self):
        config = super().get_config()
        config.update({
            "output_dim": self.output_dim,
            "num_hidden_layers": self.num_hidden_layers,
            "num_neurons_per_layer": self.hidden[0].units, # Accessing units from the first layer
            "activation": self.hidden[0].activation.__name__, # Accessing activation name
            "kernel_initializer": self.hidden[0].kernel_initializer.__class__.__name__,
        })
        return config

    def __init__(self,
            output_dim=out_dim,
            num_hidden_layers=4, 
            num_neurons_per_layer=20,
            activation='tanh',
            kernel_initializer='glorot_normal',
            **kwargs):
        super().__init__(**kwargs)

        self.num_hidden_layers = num_hidden_layers
        self.output_dim = output_dim
        self.lambd_list = []
    
        # Define NN architecture
        self.hidden = [tf.keras.layers.Dense(num_neurons_per_layer,
                             activation=tf.keras.activations.get(activation),
                             kernel_initializer=kernel_initializer)
                           for _ in range(self.num_hidden_layers)]
        self.out = tf.keras.layers.Dense(output_dim)
        
    
    def call(self, X):
        """Forward-pass through neural network."""
        Z = X
        for i in range(self.num_hidden_layers):
            Z = self.hidden[i](Z)
        return self.out(Z)

### Modifying the NN training procedure
The functions `update_last_n_losses()` and `ES()` are defined to provide early stopping of training once the relative error in the last 200 iterations is not changing below a threshold, i.e., $1\times10^{-4}$.

The function `loss_fn()` provides the MFNN architecture and error heuristics; see the above image and follow along the `loss_fn()` lines. L2 norms are necessary to control overfitting and regularize the model.

The function `get_grad()` records the trainable variables of all three NNs, calculates the loss (`loss_frac` is just a sanity check to track loss components), takes the gradient of the loss w.r.t. the trainable variables, and return those gradients.

The function `train_step()` calls the `get_grad()` function, applies the optimizer to those gradients, and returns the loss.

Finally, a `for` loop with call the `train_step()` function for a set number of iterations (`N`). Note that the `max_relative_error` variable will terminate the training loop once the early stopping procedure is triggered.

The `callback()` is responsible for printing the loss values, while `plot_loss_history()` provides the loss history w.r.t. iterations.

In this cell, it is also possible to use the `L-BFGS` optimizer. Actually, the current version uses the Adam optimizer for a set number of iterations, then use the trained weights to start an `L-BFGS` session. Two things: `L-BFGS` accepts float64 variables, and they should be flattened.

In [ ]:
class PINNSolver():
    def __init__(self, model_LF, model_HF_nl, model_HF_l, lambda_hf=1.0):  # Add weighting parameter to prioritize HF loss
        self.model_LF = model_LF
        self.model_HF_nl = model_HF_nl
        self.model_HF_l = model_HF_l
        self.lambda_hf = lambda_hf
        # Initialize history of losses and global iteration counter
        self.hist = []
        self.iter = 0
        self.last_n_losses = []
        
    def update_last_n_losses(self, loss):
        self.last_n_losses.append(loss)
        if len(self.last_n_losses) > 200:
            self.last_n_losses.pop(0)
            
    def ES(self):
        if len(self.last_n_losses) < 200:
            return 100  # a large number

        current_loss = self.last_n_losses[-1]
        max_relative_error = 100.*max([abs(current_loss - loss) / current_loss + 1e-8 for loss in self.last_n_losses[:-1]])
        return max_relative_error
    
    def loss_fn(self, X_data_LF, X_data_HF, y_data_LF, y_data_HF):
        # Forward Pass        
        y_pred_LF = self.model_LF(X_data_LF)
        y_pred_LF_HF = self.model_LF(X_data_HF)

        # HF Predictions
        input_HF = tf.concat([X_data_HF, y_pred_LF_HF], axis=1)
        y_pred_HF_nl = self.model_HF_nl(input_HF)
        y_pred_HF_l = self.model_HF_l(input_HF)
        y_pred_HF = y_pred_HF_nl + y_pred_HF_l
        
        # Regularization
        Loss_L2 = 1e-4*tf.add_n([tf.nn.l2_loss(w_) for w_ in self.model_HF_nl.trainable_weights])
        Loss_L2 += 1e-6*tf.add_n([tf.nn.l2_loss(w_) for w_ in self.model_LF.trainable_weights])
        Loss_L2 += 1e-5*tf.add_n([tf.nn.l2_loss(w_) for w_ in self.model_HF_l.trainable_weights])

        # Weighted loss
        Loss_data_LF = tf.reduce_mean(tf.square(y_data_LF - y_pred_LF))
        Loss_data_HF = tf.reduce_mean(tf.square(y_data_HF - y_pred_HF))
        loss = Loss_data_LF + (self.lambda_hf * Loss_data_HF) + Loss_L2
        loss_frac = [Loss_data_LF, Loss_data_HF, Loss_L2]
        return loss, loss_frac
    
    def get_grad(self, X_data_LF, X_data_HF, y_data_LF, y_data_HF):
        trainable_list = self.model_LF.trainable_variables + self.model_HF_nl.trainable_variables + self.model_HF_l.trainable_variables
        with tf.GradientTape(persistent=True) as tape:
            tape.watch(trainable_list)
            loss, loss_frac = self.loss_fn(X_data_LF, X_data_HF, y_data_LF, y_data_HF)   
        g = tape.gradient(loss, trainable_list)
        del tape        
        return loss, g, loss_frac
   
    
    def solve_with_TFoptimizer(self, optimizer, X_data_LF, X_data_HF, y_data_LF, y_data_HF, N=1001):
        @tf.function
        def train_step():
            loss, g, loss_frac = self.get_grad(X_data_LF, X_data_HF, y_data_LF, y_data_HF)
            trainable_list = self.model_LF.trainable_variables + self.model_HF_nl.trainable_variables + self.model_HF_l.trainable_variables
            optimizer.apply_gradients(zip(g, trainable_list))
            return loss, loss_frac
        
        for i in range(N):          
            loss, loss_frac = train_step()
            self.loss_frac = loss_frac
            self.current_loss = loss.numpy()
            self.max_relative_error = self.ES()
            self.callback(self.max_relative_error)  # Pass max_relative_error to the callback function
            self.update_last_n_losses(self.current_loss)

            if self.max_relative_error < 1e-4: # in %
                tf.print('Early stopping... \nIt {:05,d}: Loss = {:10.4e}, Max. rel. error = {} %'.format(self.iter,
                                                             self.current_loss,
                                                            np.round(self.max_relative_error, 3)))
                break
        
    def callback(self, xr=None):
        if self.iter % 5000 == 0:
            tf.print('It {:05,d}: Loss = {:10.4e}, Max. rel. error = {} %'.format(self.iter,
                                                                     self.current_loss,
                                                                     np.round(self.max_relative_error, 2)))

        self.hist.append(self.current_loss)
        self.iter+=1
        
        
    def solve_with_ScipyOptimizer(self, X_data_LF, X_data_HF, y_data_LF, y_data_HF, method='L-BFGS-B', **kwargs):
        
        # 1. Get initial weights as float64 (required by Scipy)
        def get_weight_tensor():
            weight_list = []
            shape_list = []
            # Collect all trainable variables
            trainable_vars = (self.model_LF.trainable_variables + 
                              self.model_HF_nl.trainable_variables + 
                              self.model_HF_l.trainable_variables)
            
            for v in trainable_vars:
                shape_list.append(v.shape)
                # Flatten and append
                weight_list.extend(v.numpy().flatten())
            
            # Convert to float64 for SciPy
            weight_list = np.array(weight_list, dtype=np.float64)
            return weight_list, shape_list

        x0, shape_list = get_weight_tensor()

        # 2. Function to set weights back into TF models (handles casting to float32)
        def set_weight_tensor(weight_list):
            # Ensure input is float64, but we cast to DTYPE (float32) for TF
            idx = 0
            trainable_vars = (self.model_LF.trainable_variables + 
                              self.model_HF_nl.trainable_variables + 
                              self.model_HF_l.trainable_variables)

            for v in trainable_vars:
                vs = v.shape
                
                # Determine size of this variable
                if len(vs) == 2:
                    sw = vs[0] * vs[1]
                    new_val = weight_list[idx:idx+sw].reshape((vs[0], vs[1]))
                elif len(vs) == 1:
                    sw = vs[0]
                    new_val = weight_list[idx:idx+sw]
                elif len(vs) == 0: # Scalar
                    sw = 1
                    new_val = weight_list[idx]
                
                # CRITICAL: Cast the float64 (SciPy) value to float32 (TF)
                v.assign(tf.cast(new_val, DTYPE))
                idx += sw

        # 3. The Loss and Gradient function (Interface between SciPy and TF)
        def get_loss_and_grad(w):
            # Set weights (float64 -> float32 happen inside)
            set_weight_tensor(w)
            
            # Get gradients in float32 (TF native)
            loss, grad, loss_frac = self.get_grad(X_data_LF, X_data_HF, y_data_LF, y_data_HF)
            
            # Store history
            loss_val = loss.numpy().astype(np.float64)
            self.current_loss = loss_val
            self.loss_frac = loss_frac
            self.max_relative_error = self.ES()
            
            # Flatten gradients and cast to float64
            grad_flat = []
            for g in grad:
                grad_flat.extend(g.numpy().flatten())
            
            grad_flat = np.array(grad_flat, dtype=np.float64)
            
            return loss_val, grad_flat
         
        # Run optimization
        return scipy.optimize.minimize(fun=get_loss_and_grad,
                                       x0=x0,
                                       jac=True,
                                       method=method,
                                       callback=self.callback,
                                       **kwargs)
        
    def plot_loss_history(self, ax=None):
        if not ax:
            fig = plt.figure(figsize=(7,5))
            ax = fig.add_subplot(111)
        ax.semilogy(range(len(self.hist)), self.hist,'k-')
        ax.set_xlabel('$n_{epoch}$')
        ax.set_ylabel('$\\phi^{n_{epoch}}$')
        return ax

### Instantiating the NNs and the solver
Here, the Lo-Fi, the nonlinear Hi-Fi, and the linear Hi-Fi NNs are instantiated from the `PINN_NeuralNet()` class. There are then built with their specified input shapes. We had `n=3`, which is `in_dim` in the below cell. We also have a 1-D output, which is the LAOS shear stress. Therefore, the Lo-Fi NN takes `in_dim=3`-D inputs and spits `out_dim=1`-D output. Then, the nonlinear Hi-Fi NN lifts the `out_dim=1`-D Lo-Fi output + its own inputs, which are `in_dim=3`-D; same thing for the nonlinear Hi-Fi data. That's why the Hi-Fi NNs accept `in_dim+out_dim`-D inputs.

In [ ]:
model_LF = PINN_NeuralNet(output_dim=out_dim,
                          num_hidden_layers=4,
                          num_neurons_per_layer=20)
model_HF_nl = PINN_NeuralNet(output_dim=out_dim,
                             num_hidden_layers=3,
                             num_neurons_per_layer=15)
model_HF_l = PINN_NeuralNet(output_dim=out_dim,
                            num_hidden_layers=1,
                            num_neurons_per_layer=10,
                            activation='linear')

model_LF.build(input_shape=(None,in_dim))
model_HF_nl.build(input_shape=(None,in_dim+out_dim))
model_HF_l.build(input_shape=(None,in_dim+out_dim))

solver = PINNSolver(model_LF, model_HF_nl, model_HF_l)
if 'runtime' in globals():
    del runtime

### MFNN training
We almost have everything by now. Next, we will instantiate the optimizer from `tf.keras.optimizers.legacy.Adam` with a specified learning rate. A piecewise learning rate is also provided. Then, the optimizer is called on `solver` with appropriate datasets. The `solve_with_ScipyOptimizer()` optimizer is also called after `Adam`. The latter step is optional.

In [ ]:
# ── LF Pre-training ──────────────────────────────────────────────
print("Pre-training LF network...")
optim_lf = tf.keras.optimizers.legacy.Adam(learning_rate=1e-3)

@tf.function
def pretrain_step():
    with tf.GradientTape() as tape:
        y_pred_lf = model_LF(X_data_LF)
        loss_lf = tf.reduce_mean(tf.square(y_data_LF - y_pred_lf))
        loss_lf += 1e-6 * tf.add_n([tf.nn.l2_loss(w) for w in model_LF.trainable_weights])
    grads = tape.gradient(loss_lf, model_LF.trainable_variables)
    optim_lf.apply_gradients(zip(grads, model_LF.trainable_variables))
    return loss_lf

for step in range(5000):
    loss = pretrain_step()
    if step % 1000 == 0:
        print(f"  Step {step}: LF loss = {loss.numpy():.6e}")

print("LF pre-training complete. Starting joint training...\n")

In [ ]:
lr = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
    [5000, 15000], [1e-3, 5e-4, 1e-4]
)
N_adam = 50000
N_lbfgs_maxiter = 5000    # actual stopping limit, not a safety net
N_lbfgs_maxfun  = 15000   # ~3x maxiter is a reasonable ratio

if 'runtime' not in globals():
    runtime = 0.

# ── Adam phase ────────────────────────────────────────────────────────────────
adam_complete = False
try:
    t0 = time()
    optim = tf.keras.optimizers.legacy.Adam(learning_rate=lr)
    solver.solve_with_TFoptimizer(
        optim, X_data_LF, X_data_HF, y_data_LF, y_data_HF, N=N_adam
    )
    runtime += (time() - t0) / 60.
    adam_complete = True
    print(f'\nAdam complete. Runtime: {runtime:.3f} min')
except KeyboardInterrupt:
    runtime += (time() - t0) / 60.
    print(f'\nAdam interrupted at iter {solver.iter}. Runtime: {runtime:.3f} min')

# ── L-BFGS-B phase ────────────────────────────────────────────────────────────
print(f'\nStarting L-BFGS-B from Adam checkpoint (loss={solver.current_loss:.4e})...')
try:
    t0 = time()
    result = solver.solve_with_ScipyOptimizer(
        X_data_LF, X_data_HF, y_data_LF, y_data_HF,
        method='L-BFGS-B',
        options={
            'maxiter': N_lbfgs_maxiter,
            'maxfun':  N_lbfgs_maxfun,
            'maxcor':  50,       # ← was 1000; 10–50 is optimal for NNs
            'maxls':   30,       # ← was 1000; 20–50 is sufficient
            'ftol':    1e-7,     # ← was ~2e-16; achievable with float32 precision
            'gtol':    1e-5,     # gradient norm stopping — add this!
        }
    )
    runtime += (time() - t0) / 60.
    print(f'\nL-BFGS-B finished. Status: {result.message}')
    print(f'Iterations: {result.nit} | Fun evals: {result.nfev}')
    print(f'Runtime: {(time()-t0)/60.:.3f} min | Total: {runtime:.3f} min')
except KeyboardInterrupt:
    runtime += (time() - t0) / 60.
    print(f'\nL-BFGS-B interrupted. Runtime: {runtime:.3f} min')

print(f'\nFinal loss components: {np.array(solver.loss_frac)}')

### Visualizing the results
We again need the raw data to plot the Hi-Fi data for each set of experiments. So, the same functions we used to generate the Lo-Fi data are brought here.

In [ ]:
def pos_finder(vector):
    vector = np.array(vector)
    for i in range(len(vector) - 1):
        if vector[i] < 0 and vector[i + 1] >= 0:
            return i + 1
    return np.argmin(np.abs(vector))

path = os.getcwd()
files = os.listdir(path)
df = {}
df = pd.read_excel(url_hf_raw, sheet_name=None)
for v in df.values():
    v["Shear rate"] = v["Shear rate"] * v["Angular frequency"]
data = [
    [k, v] for k, v in df.items() 
    if (v["Oscillation strain"].iloc[0] <= 4) & (v["Oscillation strain"].iloc[0] >= 0.00)
]  # NOTE: Adjust to include different sheets

t_train, g_train, w_train = np.array([]), np.array([]), np.array([])
s_train = np.array([])

def extractor(i):
    data[i][1] = data[i][1].dropna()
    print(data[i][0])
    
    # Handling the time shift
    ind = pos_finder(data[i][1]['Strain'])
    print("Start Index: ", ind)
    t0 = data[i][1]['Step time'].iloc[ind]
    ymax = data[i][1]['Stress'].iloc[ind]
    
    # Column extraction 
    St = data[i][1]['Strain']
    SR = data[i][1]['Shear rate']
    Time = data[i][1]['Step time']
    w = data[i][1]['Angular frequency']
    G0 = data[i][1]['Oscillation strain']
    Stress = data[i][1]['Stress']
    Time_adj = Time - t0
    Time_adj = Time_adj
    return Time_adj, G0, w, Stress, St, SR

### Generating stress vs. strain curves for all Lo-Fi cases.

In [ ]:
now = datetime.now().strftime("%Y_%m_%d-%H_%M_%S")

# Ensure 'data' is loaded correctly (relying on the previous block)
# data was defined as: data = [[k,v] for k,v in df.items()]

print(f"Found {len(data)} sheets of data. processing...")

with pd.ExcelWriter(now+".xlsx") as writer:
    for i in range(len(data)):
        try:
            t_test, g0_test, w_test, s_test, st_test, sr_test = extractor(i)  # TODO: Implement dictionary fix like on other notebook
            g0 = np.unique(g0_test)[0]
            omega = np.unique(w_test)[0]

            # 1. Stack inputs including G0
            X_test = np.column_stack([st_test, sr_test, w_test, g0_test])
            y_test = np.column_stack([s_test])

            # 2. Normalize 
            X_test_norm, y_test_norm = normalize_data(X_test, y_test, scaler_X, scaler_y)
            
            # 3. Predict
            y_LF = model_LF(X_test_norm)
            X_MF = np.column_stack((X_test_norm, y_LF))
            y_MF = model_HF_nl(X_MF) + model_HF_l(X_MF)
            
            # 4. Denormalize 
            y_LF_np = y_LF.numpy() if hasattr(y_LF, 'numpy') else np.array(y_LF)
            y_MF_np = y_MF.numpy() if hasattr(y_MF, 'numpy') else y_MF
            
            y_LF_denorm = scaler_y.inverse_transform(y_LF_np)
            y_MF_denorm = scaler_y.inverse_transform(y_MF_np)
            
            # Plots
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.scatter(st_test, y_test, label='ref', color='k', s=2, alpha=0.3)
            ax.plot(st_test, y_MF_denorm, label='mf', color='tab:red', zorder=10, lw=2)
            ax.plot(st_test, y_LF_denorm, label='LF (Maxwell)',
                    color='tab:blue', lw=1.5, linestyle='--', zorder=2, alpha=0.7)
            plt.xlabel('Strain')
            plt.ylabel('Stress')
            plt.title(f"Sheet {i} | w={omega:.6f}, g0={g0:.6f}")
            plt.legend()
            
            # Optional: If you have >20 sheets, uncomment the line below 
            # to prevent 50+ plots from crashing your notebook memory:
            # plt.close(fig) 
            plt.show()
            
            # --- SAVING TO EXCEL ---
            df_exp = np.column_stack([t_test, g0_test, w_test, st_test, sr_test, s_test, y_MF_denorm])
            df_exp = pd.DataFrame(df_exp)
            df_exp.columns = ['Time', 'G0', 'AngFreq', 'Strain', 'StrainRate', 'Stress', 'Stress_pred']
            
            # Excel sheet names have a 31 char limit, truncate if necessary
            safe_sheet_name = str(i)[:31]
            df_exp.to_excel(writer, sheet_name=safe_sheet_name, index=False)
            
        except Exception as e:
            print(f"Skipping sheet index {i} due to error: {e}")

In [ ]:
import shutil
import joblib # Ensure joblib is imported

# Choose directory
save_dir = r"/home/alanh/projects/PunLab/mfnn/models/pig_pstat"
os.makedirs(save_dir, exist_ok=True) # Good practice to ensure the dir exists

# Save the three sub-models
model_LF.save(os.path.join(save_dir, 'model_LF'))
model_HF_nl.save(os.path.join(save_dir, 'model_HF_nl'))
model_HF_l.save(os.path.join(save_dir, 'model_HF_l'))

# Save the Scalers instead of the normalization vector
joblib.dump(scaler_X, os.path.join(save_dir, 'scaler_X.pkl'))
joblib.dump(scaler_y, os.path.join(save_dir, 'scaler_y.pkl'))

print(f"Models and scalers successfully saved to: {save_dir}")

In [ ]:
# Plot without using experimental st_test and sr_test!

test_amps = [0.01, 0.05, 0.1, 0.5, 1.0, 2.687103]

for i in test_amps:

    t_test, g0_test, w_test, s_test, st_test, sr_test = extractor(1)

    g0_test = g0_test / g0_test * i

    st_test = g0_test * np.sin(w_test * t_test)
    sr_test = g0_test * w_test * np.cos(w_test * t_test)

    g0 = np.unique(g0_test)[0]
    omega = np.unique(w_test)[0]

    # 1. Stack inputs including G0
    X_test = np.column_stack([st_test, sr_test, w_test, g0_test])
    y_test = np.column_stack([s_test])

    # 2. Normalize 
    X_test_norm, y_test_norm = normalize_data(X_test, y_test, scaler_X, scaler_y)

    # 3. Predict
    y_LF = model_LF(X_test_norm)
    X_MF = np.column_stack((X_test_norm, y_LF))
    y_MF = model_HF_nl(X_MF) + model_HF_l(X_MF)

    # 4. Denormalize 
    y_MF_denorm = scaler_y.inverse_transform(y_MF.numpy())

    # --- PLOTTING ---
    fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(st_test, y_MF_denorm, label='mf', color='tab:red', zorder=10, lw=2)
    plt.xlabel('Strain')
    plt.ylabel('Stress')
    plt.title(f"Sheet {i} | w={omega:.6f}, g0={g0:.6f}")
    plt.legend()



In [ ]:
# Plot at w = 0.628rad, g0 = 0.023593 using CALCULATED inputs

# 1. Extract the raw data containers first
t_test, g0_test, w_test, s_test, st_test_exp, sr_test_exp = extractor(5)

# 2. Get scalar values for the equation
g0 = np.max(g0_test) 
omega = np.max(w_test)

print(f"Calculating with G0: {g0} and w: {omega}")

# 3. Calculate analytical Strain and Rate
# We use t_test (which is Time_adj) from the extractor
st_test_calc = g0 * np.sin(omega * t_test)
sr_test_calc = g0 * omega * np.cos(omega * t_test) # NOTE: lf & hf data have strain rate divided by omega. omega is constant

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(t_test[:100], st_test_exp[:100], 'k-', label='Experimental', alpha=0.5)
plt.plot(t_test[:100], st_test_calc[:100], 'r--', label='Calculated')
plt.title('Strain Comparison (First 100 pts)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(t_test[:100], sr_test_exp[:100], 'k-', label='Experimental', alpha=0.5)
plt.plot(t_test[:100], sr_test_calc[:100], 'r--', label='Calculated')
plt.title('Rate Comparison (First 100 pts)')
plt.legend()
plt.show()

w_vec = np.full_like(st_test_calc, omega)
g0_vec = np.full_like(st_test_calc, g0)

X_test = np.column_stack([st_test_calc, sr_test_calc, w_vec, g0_vec])
y_test = np.column_stack([s_test]) # Ground truth stress

# Normalize 
X_test_norm, y_test_norm = normalize_data(X_test, y_test, scaler_X, scaler_y)

# Predict
y_LF = model_LF(X_test_norm)
X_MF = np.column_stack((X_test_norm, y_LF))
y_MF = model_HF_nl(X_MF) + model_HF_l(X_MF)

# Denormalize 
y_MF_denorm = scaler_y.inverse_transform(y_MF.numpy())

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(st_test_calc, y_MF_denorm, label='MFNN (Calc Input)', color='tab:red', zorder=10, lw=2)
ax.plot(st_test_exp, s_test, label='Experimental Truth', color='k', alpha=0.2)

plt.xlabel('Strain')
plt.ylabel('Stress')
plt.title(f"Sheet 5 | w={omega:.4f}, g0={g0:.4f}")
plt.legend()
plt.show()

### Predict Test Data

In [ ]:
path = r"/home/alanh/projects/PunLab/mfnn/experiments/test_data/TJC2_047_B.xlsx"
df = pd.read_excel(path, sheet_name=None)
for v in df.values():
    v["Shear rate"] = v["Shear rate"] * v["Angular frequency"]
data = [
    [k, v] for k, v in df.items() 
    if (v["Oscillation strain"].iloc[0] <= 4) & (v["Oscillation strain"].iloc[0] >= 0.00)
]  # NOTE: Adjust to include different sheets

t_train, g_train, w_train = np.array([]), np.array([]), np.array([])
s_train = np.array([])

for i in range(len(data)):
    t_test, g0_test, w_test, s_test, st_test, sr_test = extractor(i)
    g0 = np.unique(g0_test)[0]
    omega = np.unique(w_test)[0]

    st_test = g0 * np.sin(omega * t_test)
    sr_test = g0 * omega * np.cos(omega * t_test)

    X_test = np.column_stack([st_test, sr_test, w_test, g0_test])
    y_test = np.column_stack([s_test])  # NOTE: Use calculated y_test, we just need this for normalization function

    X_test_norm, y_test_norm = normalize_data(X_test, y_test, scaler_X, scaler_y)

    y_LF = model_LF(X_test_norm)
    X_MF = np.column_stack((X_test_norm, y_LF))
    y_MF = model_HF_nl(X_MF) + model_HF_l(X_MF)

    y_LF_np = y_LF.numpy() if hasattr(y_LF, 'numpy') else np.array(y_LF)
    y_MF_np = y_MF.numpy() if hasattr(y_MF, 'numpy') else y_MF
    
    y_LF_denorm = scaler_y.inverse_transform(y_LF_np)
    y_MF_denorm = scaler_y.inverse_transform(y_MF_np)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(st_test, y_test, label='ref', color='k', s=2, alpha=0.3)
    ax.plot(st_test, y_MF_denorm, label='mf', color='tab:red', zorder=10, lw=2)
    ax.plot(st_test, y_LF_denorm, label='LF (Maxwell)',
            color='tab:blue', lw=1.5, linestyle='--', zorder=2, alpha=0.7)
    plt.xlabel('Strain')
    plt.ylabel('Stress')
    plt.title(f"Sheet {i} | w={omega:.6f}, g0={g0:.6f}")
    plt.legend()
    plt.show()




In [ ]:
"""
MFNN LAOS Rheometry — Comprehensive Diagnostics
=================================================
Run this script AFTER Notebook 1 (LF generation) has been executed,
so that all variables (data, X_fit_dict, Y_fit_dict, y0_fit_dict,
gpr_eta, gpr_tau, etc.) are in scope.

Copy-paste each section into a notebook cell, or run the whole file
after exec-ing Notebook 1's namespace into globals().

Diagnostics covered:
  1. Maxwell fit quality per sheet (R², η/τ vs γ₀, overlay grid)
  2. GPR extrapolation quality (LOO CV, predicted vs actual)
  3. LF vs HF stress amplitude ratio
  4. Lissajous antisymmetry check
  5. Normalized input distribution divergence (LF vs HF)
  6. Per-sheet loss contribution (after MFNN training)
  7. pos_finder validation
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.integrate import odeint
from scipy.interpolate import interp1d
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.model_selection import LeaveOneOut
import warnings
warnings.filterwarnings("ignore")


# ═══════════════════════════════════════════════════════════════════
# HELPER: Ensure the Notebook-1 namespace is available
# ═══════════════════════════════════════════════════════════════════
# If running as a standalone script, you must first execute Notebook 1
# and then pass its namespace. In a notebook, just paste each section
# into cells after running Notebook 1.


# ═══════════════════════════════════════════════════════════════════
# DIAGNOSTIC 1 — Maxwell fit quality per sheet
# ═══════════════════════════════════════════════════════════════════
def diagnostic_1_maxwell_fit_quality(data, X_fit_dict, Y_fit_dict, y0_fit_dict, extractor, fit_function):
    """
    For every sheet:
      • Compute R² of the Maxwell ODE fit on the stress waveform
      • Plot η vs γ₀ and τ vs γ₀ on log-log axes
      • Overlay fitted Maxwell loop vs reference loop in a grid
    """
    n_sheets = len(data)
    
    etas = np.array([Y_fit_dict[i][0] for i in range(n_sheets)])
    taus = np.array([Y_fit_dict[i][1] for i in range(n_sheets)])
    g0s  = np.array([X_fit_dict[i][1] for i in range(n_sheets)])
    omegas = np.array([X_fit_dict[i][0] for i in range(n_sheets)])
    
    r2_scores = []
    stress_amp_ratios = []
    
    print("=" * 100)
    print("DIAGNOSTIC 1: Maxwell Fit Quality Per Sheet")
    print("=" * 100)
    print(f"{'Sheet':>5} {'γ₀':>10} {'ω':>8} {'η':>10} {'τ':>10} {'R²':>8} "
          f"{'|σ_fit|_max':>12} {'|σ_ref|_max':>12} {'Amp Ratio':>10} {'FLAG':>8}")
    print("-" * 100)
    
    for i in range(n_sheets):
        t_i, g0_i, w_i, s_i, st_i, sr_i = extractor(i)
        eta_i, tau_i = Y_fit_dict[i]
        omega_i, g0_val = X_fit_dict[i]
        y0_i = y0_fit_dict[i]
        
        # Compute Maxwell prediction via ODE
        s_i_np = s_i.values if hasattr(s_i, 'values') else np.array(s_i)
        t_i_np = t_i.values if hasattr(t_i, 'values') else np.array(t_i)
        
        try:
            s_pred = fit_function(t_i_np, eta_i, tau_i, y0_i, g0_val, omega_i)
            r2 = r2_score(s_i_np, s_pred.flatten())
        except Exception as e:
            r2 = float('nan')
            s_pred = np.zeros_like(s_i_np)
        
        s_fit_max = np.max(np.abs(s_pred))
        s_ref_max = np.max(np.abs(s_i_np))
        amp_ratio = s_fit_max / (s_ref_max + 1e-15)
        
        flag = ""
        if r2 < 0.5:
            flag += " LOW_R²"
        if amp_ratio > 2.0 or amp_ratio < 0.5:
            flag += " AMP_MISMATCH"
        if eta_i > 1000 or eta_i < 1e-4:
            flag += " η_EXTREME"
        if tau_i > 100 or tau_i < 1e-4:
            flag += " τ_EXTREME"
        
        r2_scores.append(r2)
        stress_amp_ratios.append(amp_ratio)
        
        print(f"{i:>5} {g0_val:>10.6f} {omega_i:>8.4f} {eta_i:>10.4f} {tau_i:>10.4f} {r2:>8.4f} "
              f"{s_fit_max:>12.4f} {s_ref_max:>12.4f} {amp_ratio:>10.3f} {flag}")
    
    # ── Plot η and τ vs γ₀ ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ax = axes[0]
    ax.loglog(g0s, etas, 'ko-', markersize=6)
    for i in range(n_sheets):
        if r2_scores[i] < 0.5:
            ax.loglog(g0s[i], etas[i], 'rs', markersize=12, markerfacecolor='none', linewidth=2)
    ax.set_xlabel('γ₀ (strain amplitude)')
    ax.set_ylabel('η (viscosity)')
    ax.set_title('Fitted η vs γ₀ (red squares = R² < 0.5)')
    ax.grid(True, alpha=0.3)
    
    ax = axes[1]
    ax.loglog(g0s, taus, 'ko-', markersize=6)
    for i in range(n_sheets):
        if r2_scores[i] < 0.5:
            ax.loglog(g0s[i], taus[i], 'rs', markersize=12, markerfacecolor='none', linewidth=2)
    ax.set_xlabel('γ₀ (strain amplitude)')
    ax.set_ylabel('τ (relaxation time)')
    ax.set_title('Fitted τ vs γ₀ (red squares = R² < 0.5)')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('diag1_eta_tau_vs_g0.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # ── Grid of Lissajous overlays (Maxwell fit vs reference) ──
    n_cols = 5
    n_rows = int(np.ceil(n_sheets / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows))
    axes = axes.flatten()
    
    for i in range(n_sheets):
        ax = axes[i]
        t_i, g0_i, w_i, s_i, st_i, sr_i = extractor(i)
        eta_i, tau_i = Y_fit_dict[i]
        omega_i, g0_val = X_fit_dict[i]
        y0_i = y0_fit_dict[i]
        
        s_i_np = s_i.values if hasattr(s_i, 'values') else np.array(s_i)
        st_i_np = st_i.values if hasattr(st_i, 'values') else np.array(st_i)
        t_i_np = t_i.values if hasattr(t_i, 'values') else np.array(t_i)
        
        try:
            s_pred = fit_function(t_i_np, eta_i, tau_i, y0_i, g0_val, omega_i)
            g_pred = g0_val * np.sin(omega_i * t_i_np)
        except:
            s_pred = np.zeros_like(s_i_np)
            g_pred = np.zeros_like(s_i_np)
        
        ax.scatter(st_i_np, s_i_np, s=1, color='k', alpha=0.2, label='HF ref')
        ax.plot(g_pred, s_pred.flatten(), 'r-', linewidth=1.5, label='Maxwell')
        ax.set_title(f"#{i} γ₀={g0_val:.4f}\nR²={r2_scores[i]:.3f}", fontsize=8)
        ax.tick_params(labelsize=6)
        if i == 0:
            ax.legend(fontsize=6)
    
    for j in range(n_sheets, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('DIAGNOSTIC 1: Maxwell Fit vs Reference (All Sheets)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig('diag1_maxwell_grid.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return r2_scores, stress_amp_ratios


# ═══════════════════════════════════════════════════════════════════
# DIAGNOSTIC 2 — GPR extrapolation quality
# ═══════════════════════════════════════════════════════════════════
def diagnostic_2_gpr_quality(X_fit_dict, Y_fit_dict, gpr_eta, gpr_tau, X_log, Y_log):
    """
    LOO cross-validation on η and τ GPR.
    Plot predicted vs actual on log-log axes.
    Flag sheets where GPR relative error > 30%.
    """
    n_sheets = len(X_fit_dict)
    
    print("\n" + "=" * 100)
    print("DIAGNOSTIC 2: GPR Extrapolation Quality (LOO)")
    print("=" * 100)
    
    loo = LeaveOneOut()
    eta_pred_loo = np.zeros(n_sheets)
    tau_pred_loo = np.zeros(n_sheets)
    eta_std_loo = np.zeros(n_sheets)
    tau_std_loo = np.zeros(n_sheets)
    
    # Use the already-optimized kernels but refit in LOO
    for train_idx, test_idx in loo.split(X_log):
        gpr_eta_tmp = GaussianProcessRegressor(
            kernel=gpr_eta.kernel_, optimizer=None, normalize_y=True, random_state=8
        )
        gpr_tau_tmp = GaussianProcessRegressor(
            kernel=gpr_tau.kernel_, optimizer=None, normalize_y=True, random_state=8
        )
        
        gpr_eta_tmp.fit(X_log[train_idx], Y_log[train_idx, 0])
        gpr_tau_tmp.fit(X_log[train_idx], Y_log[train_idx, 1])
        
        mu_e, std_e = gpr_eta_tmp.predict(X_log[test_idx], return_std=True)
        mu_t, std_t = gpr_tau_tmp.predict(X_log[test_idx], return_std=True)
        
        eta_pred_loo[test_idx] = mu_e
        tau_pred_loo[test_idx] = mu_t
        eta_std_loo[test_idx] = std_e
        tau_std_loo[test_idx] = std_t
    
    # Convert back to original space
    eta_actual = np.exp(Y_log[:, 0])
    tau_actual = np.exp(Y_log[:, 1])
    eta_pred_orig = np.exp(eta_pred_loo)
    tau_pred_orig = np.exp(tau_pred_loo)
    
    g0s = np.array([X_fit_dict[i][1] for i in range(n_sheets)])
    
    print(f"\n{'Sheet':>5} {'γ₀':>10} {'η_actual':>10} {'η_pred':>10} {'η_err%':>8} {'η_GPR_σ':>8} "
          f"{'τ_actual':>10} {'τ_pred':>10} {'τ_err%':>8} {'τ_GPR_σ':>8} {'FLAG':>12}")
    print("-" * 115)
    
    for i in range(n_sheets):
        eta_err = 100 * abs(eta_pred_orig[i] - eta_actual[i]) / (eta_actual[i] + 1e-15)
        tau_err = 100 * abs(tau_pred_orig[i] - tau_actual[i]) / (tau_actual[i] + 1e-15)
        flag = ""
        if eta_err > 30:
            flag += " η>30%"
        if tau_err > 30:
            flag += " τ>30%"
        if eta_std_loo[i] > 0.5:
            flag += " η_UNCERT"
        if tau_std_loo[i] > 0.5:
            flag += " τ_UNCERT"
        
        print(f"{i:>5} {g0s[i]:>10.6f} {eta_actual[i]:>10.4f} {eta_pred_orig[i]:>10.4f} "
              f"{eta_err:>7.1f}% {eta_std_loo[i]:>8.4f} "
              f"{tau_actual[i]:>10.4f} {tau_pred_orig[i]:>10.4f} "
              f"{tau_err:>7.1f}% {tau_std_loo[i]:>8.4f} {flag}")
    
    # ── Predicted vs Actual plots ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for ax, actual, pred, std, label in [
        (axes[0], eta_actual, eta_pred_orig, eta_std_loo, 'η'),
        (axes[1], tau_actual, tau_pred_orig, tau_std_loo, 'τ')
    ]:
        ax.loglog(actual, pred, 'ko', markersize=8)
        # Color by γ₀
        scatter = ax.scatter(actual, pred, c=np.log10(g0s), cmap='viridis', 
                           s=60, edgecolors='k', zorder=5)
        
        # Perfect prediction line
        lims = [min(actual.min(), pred.min()) * 0.5, max(actual.max(), pred.max()) * 2]
        ax.plot(lims, lims, 'k--', alpha=0.3, label='Perfect')
        ax.fill_between(lims, [l * 0.7 for l in lims], [l * 1.3 for l in lims], 
                        alpha=0.1, color='green', label='±30%')
        
        # Flag outliers
        for i in range(n_sheets):
            err = abs(pred[i] - actual[i]) / (actual[i] + 1e-15)
            if err > 0.3:
                ax.annotate(f'{i}', (actual[i], pred[i]), fontsize=8, color='red',
                           fontweight='bold', ha='left')
        
        ax.set_xlabel(f'Actual {label}')
        ax.set_ylabel(f'GPR LOO Predicted {label}')
        ax.set_title(f'{label}: Predicted vs Actual (LOO)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=ax, label='log₁₀(γ₀)')
    
    plt.tight_layout()
    plt.savefig('diag2_gpr_loo.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # ── GPR uncertainty vs γ₀ ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].semilogx(g0s, eta_std_loo, 'bo-')
    axes[0].axhline(0.5, color='r', linestyle='--', label='High uncertainty threshold')
    axes[0].set_xlabel('γ₀'); axes[0].set_ylabel('GPR σ (log-space)')
    axes[0].set_title('η GPR Uncertainty vs γ₀'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    
    axes[1].semilogx(g0s, tau_std_loo, 'bo-')
    axes[1].axhline(0.5, color='r', linestyle='--', label='High uncertainty threshold')
    axes[1].set_xlabel('γ₀'); axes[1].set_ylabel('GPR σ (log-space)')
    axes[1].set_title('τ GPR Uncertainty vs γ₀'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('diag2_gpr_uncertainty.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return eta_pred_orig, tau_pred_orig, eta_std_loo, tau_std_loo


# ═══════════════════════════════════════════════════════════════════
# DIAGNOSTIC 3 — LF vs HF stress amplitude ratio
# ═══════════════════════════════════════════════════════════════════
def diagnostic_3_lf_hf_amplitude_ratio(data, extractor, X_fit_dict, Y_fit_dict, y0_fit_dict,
                                        gpr_eta, gpr_tau):
    """
    For each sheet, compute ratio = max(|σ_LF|) / max(|σ_HF|).
    Flag sheets where ratio < 0.5 or > 2.0.
    """
    n_sheets = len(data)
    
    print("\n" + "=" * 100)
    print("DIAGNOSTIC 3: LF vs HF Stress Amplitude Ratio")
    print("=" * 100)
    
    ratios = []
    g0s = []
    
    print(f"\n{'Sheet':>5} {'γ₀':>10} {'ω':>8} {'η_GPR':>10} {'τ_GPR':>10} "
          f"{'|σ_LF|_max':>12} {'|σ_HF|_max':>12} {'Ratio':>8} {'FLAG':>20}")
    print("-" * 110)
    
    for i in range(n_sheets):
        t_i, g0_i, w_i, s_i, st_i, sr_i = extractor(i)
        omega_i, g0_val = X_fit_dict[i]
        y0_i = y0_fit_dict[i]
        
        # GPR-predicted η, τ
        X_pred_l = np.log(np.reshape([omega_i, g0_val], (1, -1)))
        eta_gpr = np.exp(gpr_eta.predict(X_pred_l)[0])
        tau_gpr = np.exp(gpr_tau.predict(X_pred_l)[0])
        
        # Analytical steady-state Maxwell LF stress
        G_prime = (eta_gpr * tau_gpr * omega_i**2) / (1 + (tau_gpr * omega_i)**2)
        G_double_prime = (eta_gpr * omega_i) / (1 + (tau_gpr * omega_i)**2)
        
        t_np = t_i.values if hasattr(t_i, 'values') else np.array(t_i)
        s_lf = g0_val * (G_prime * np.sin(omega_i * t_np) + G_double_prime * np.cos(omega_i * t_np))
        
        s_hf_np = s_i.values if hasattr(s_i, 'values') else np.array(s_i)
        
        lf_max = np.max(np.abs(s_lf))
        hf_max = np.max(np.abs(s_hf_np))
        ratio = lf_max / (hf_max + 1e-15)
        
        flag = ""
        if ratio > 2.0:
            flag = "⚠ LF >> HF (TOO LARGE)"
        elif ratio < 0.5:
            flag = "⚠ LF << HF (TOO SMALL)"
        elif ratio > 1.5:
            flag = "~ LF somewhat large"
        elif ratio < 0.7:
            flag = "~ LF somewhat small"
        
        ratios.append(ratio)
        g0s.append(g0_val)
        
        print(f"{i:>5} {g0_val:>10.6f} {omega_i:>8.4f} {eta_gpr:>10.4f} {tau_gpr:>10.4f} "
              f"{lf_max:>12.4f} {hf_max:>12.4f} {ratio:>8.3f} {flag}")
    
    # ── Plot ratio vs γ₀ ──
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.semilogx(g0s, ratios, 'ko-', markersize=8)
    ax.axhline(1.0, color='green', linestyle='-', alpha=0.5, label='Perfect (1.0)')
    ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Lower bound (0.5)')
    ax.axhline(2.0, color='red', linestyle='--', alpha=0.5, label='Upper bound (2.0)')
    ax.fill_between([min(g0s) * 0.5, max(g0s) * 2], 0.5, 2.0, alpha=0.05, color='green')
    
    for i in range(n_sheets):
        if ratios[i] > 2.0 or ratios[i] < 0.5:
            ax.annotate(f'{i}', (g0s[i], ratios[i]), fontsize=9, color='red', fontweight='bold')
    
    ax.set_xlabel('γ₀ (strain amplitude)')
    ax.set_ylabel('max|σ_LF| / max|σ_HF|')
    ax.set_title('DIAGNOSTIC 3: LF/HF Stress Amplitude Ratio vs γ₀')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(ratios) * 1.2)
    plt.tight_layout()
    plt.savefig('diag3_lf_hf_ratio.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return ratios


# ═══════════════════════════════════════════════════════════════════
# DIAGNOSTIC 4 — Lissajous antisymmetry check
# ═══════════════════════════════════════════════════════════════════
def diagnostic_4_antisymmetry(data, extractor):
    """
    For each sheet's reference data, compute the antisymmetry residual:
      R = σ(γ) + σ(-γ) after interpolating to a regular grid.
    Report mean|R| normalized by stress amplitude.
    """
    n_sheets = len(data)
    
    print("\n" + "=" * 100)
    print("DIAGNOSTIC 4: Lissajous Antisymmetry Check")
    print("=" * 100)
    print("  Point-antisymmetry: σ(-γ,-γ̇) = -σ(γ,γ̇)")
    print("  Residual R = [σ(γ,γ̇) + σ(-γ,-γ̇)] / (2·σ_amp)")
    print("  R ≈ 0 means perfectly antisymmetric")
    print()
    
    residuals = []
    g0s = []
    
    print(f"{'Sheet':>5} {'γ₀':>10} {'mean|R|':>10} {'max|R|':>10} {'σ_amp':>10} {'NOTE':>20}")
    print("-" * 75)
    
    for i in range(n_sheets):
        t_i, g0_i, w_i, s_i, st_i, sr_i = extractor(i)
        
        s_np = s_i.values if hasattr(s_i, 'values') else np.array(s_i)
        st_np = st_i.values if hasattr(st_i, 'values') else np.array(st_i)
        sr_np = sr_i.values if hasattr(sr_i, 'values') else np.array(sr_i)
        g0_val = np.unique(g0_i)[0] if hasattr(g0_i, 'values') else np.unique(g0_i)[0]
        
        s_amp = (np.max(s_np) - np.min(s_np)) / 2.0
        
        # Check antisymmetry by pairing (γ, γ̇, σ) with (-γ, -γ̇)
        # For oscillatory data, if γ(t) = γ₀ sin(ωt), then γ(t+π/ω) = -γ(t)
        # So we compare σ(t) with -σ(t + T/2)
        n = len(s_np)
        half = n // 2
        
        if half > 10:
            # Compare first half with negated second half
            len_compare = min(half, n - half)
            R = s_np[:len_compare] + s_np[half:half + len_compare]
            R_norm = R / (2.0 * s_amp + 1e-15)
            mean_R = np.mean(np.abs(R_norm))
            max_R = np.max(np.abs(R_norm))
        else:
            mean_R = float('nan')
            max_R = float('nan')
        
        note = ""
        if mean_R > 0.2:
            note = "⚠ SIGNIFICANT ASYMMETRY"
        elif mean_R > 0.1:
            note = "~ moderate asymmetry"
        
        residuals.append(mean_R)
        g0s.append(g0_val)
        
        print(f"{i:>5} {g0_val:>10.6f} {mean_R:>10.4f} {max_R:>10.4f} {s_amp:>10.4f} {note}")
    
    # ── Plot ──
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.semilogx(g0s, residuals, 'ko-', markersize=8)
    ax.axhline(0.1, color='orange', linestyle='--', label='Moderate threshold')
    ax.axhline(0.2, color='red', linestyle='--', label='High threshold')
    ax.set_xlabel('γ₀')
    ax.set_ylabel('Mean |R| (normalized antisymmetry residual)')
    ax.set_title('DIAGNOSTIC 4: Lissajous Antisymmetry Residual')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('diag4_antisymmetry.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return residuals


# ═══════════════════════════════════════════════════════════════════
# DIAGNOSTIC 5 — Normalized input distribution (LF vs HF)
# ═══════════════════════════════════════════════════════════════════
def diagnostic_5_input_distributions(X_data_LF, X_data_HF, y_data_LF, y_data_HF, scaler_X, scaler_y):
    """
    After applying scaler_X, plot histograms of each normalized feature
    for LF and HF data overlaid. Flag features where distributions diverge.
    
    NOTE: Pass the ALREADY-NORMALIZED tensors (as produced by Notebook 2).
    """
    print("\n" + "=" * 100)
    print("DIAGNOSTIC 5: Normalized Input Distribution (LF vs HF)")
    print("=" * 100)
    
    # Convert to numpy
    X_lf = X_data_LF.numpy() if hasattr(X_data_LF, 'numpy') else np.array(X_data_LF)
    X_hf = X_data_HF.numpy() if hasattr(X_data_HF, 'numpy') else np.array(X_data_HF)
    y_lf = y_data_LF.numpy() if hasattr(y_data_LF, 'numpy') else np.array(y_data_LF)
    y_hf = y_data_HF.numpy() if hasattr(y_data_HF, 'numpy') else np.array(y_data_HF)
    
    feature_names = ['Strain (norm)', 'Strain Rate (norm)', 'ω (norm)', 'γ₀ (norm)']
    n_features = X_lf.shape[1]
    
    fig, axes = plt.subplots(1, n_features + 1, figsize=(4 * (n_features + 1), 4))
    
    for j in range(n_features):
        ax = axes[j]
        ax.hist(X_hf[:, j], bins=50, alpha=0.5, label='HF', color='black', density=True)
        ax.hist(X_lf[:, j], bins=50, alpha=0.5, label='LF', color='blue', density=True)
        ax.set_title(feature_names[j] if j < len(feature_names) else f'Feature {j}')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        
        # Compute KL-like divergence (simple: difference of means and stds)
        mean_diff = abs(np.mean(X_lf[:, j]) - np.mean(X_hf[:, j]))
        std_ratio = np.std(X_lf[:, j]) / (np.std(X_hf[:, j]) + 1e-10)
        print(f"  {feature_names[j] if j < len(feature_names) else f'Feature {j}':>25}: "
              f"mean_diff={mean_diff:.4f}, std_ratio={std_ratio:.4f}"
              f"{'  ⚠ DIVERGENT' if mean_diff > 0.5 or std_ratio > 2.0 or std_ratio < 0.5 else ''}")
    
    # Output (stress)
    ax = axes[-1]
    ax.hist(y_hf.flatten(), bins=50, alpha=0.5, label='HF', color='black', density=True)
    ax.hist(y_lf.flatten(), bins=50, alpha=0.5, label='LF', color='blue', density=True)
    ax.set_title('Stress (norm)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    mean_diff_y = abs(np.mean(y_lf) - np.mean(y_hf))
    std_ratio_y = np.std(y_lf) / (np.std(y_hf) + 1e-10)
    print(f"  {'Stress (norm)':>25}: mean_diff={mean_diff_y:.4f}, std_ratio={std_ratio_y:.4f}"
          f"{'  ⚠ DIVERGENT' if mean_diff_y > 0.5 or std_ratio_y > 2.0 or std_ratio_y < 0.5 else ''}")
    
    plt.suptitle('DIAGNOSTIC 5: Normalized Feature Distributions (LF=blue, HF=black)', fontsize=12)
    plt.tight_layout()
    plt.savefig('diag5_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # ── Scatter: γ₀ feature distribution ──
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(X_hf[:, 0], X_hf[:, 1], s=1, alpha=0.1, c='k', label='HF')
    axes[0].scatter(X_lf[:, 0], X_lf[:, 1], s=1, alpha=0.1, c='b', label='LF')
    axes[0].set_xlabel('Strain (norm)'); axes[0].set_ylabel('Strain Rate (norm)')
    axes[0].set_title('Input space: Strain vs Strain Rate')
    axes[0].legend(markerscale=10)
    
    axes[1].scatter(X_hf[:, 2], X_hf[:, 3], s=3, alpha=0.3, c='k', label='HF')
    axes[1].scatter(X_lf[:, 2], X_lf[:, 3], s=3, alpha=0.3, c='b', label='LF')
    axes[1].set_xlabel('ω (norm)'); axes[1].set_ylabel('γ₀ (norm)')
    axes[1].set_title('Input space: ω vs γ₀')
    axes[1].legend(markerscale=5)
    
    plt.tight_layout()
    plt.savefig('diag5_input_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()


# ═══════════════════════════════════════════════════════════════════
# DIAGNOSTIC 6 — Per-sheet loss contribution (after MFNN training)
# ═══════════════════════════════════════════════════════════════════
def diagnostic_6_per_sheet_loss(data, extractor, model_LF, model_HF_nl, model_HF_l,
                                 scaler_X, scaler_y, normalize_data):
    """
    After training, compute Loss_LF, Loss_HF, Loss_total per sheet.
    Identifies which sheets are well-learned vs ignored.
    
    Pass the trained models and the normalize_data function from Notebook 2.
    """
    import tensorflow as tf
    
    n_sheets = len(data)
    
    print("\n" + "=" * 100)
    print("DIAGNOSTIC 6: Per-Sheet Loss Contribution (Post-Training)")
    print("=" * 100)
    
    print(f"\n{'Sheet':>5} {'γ₀':>10} {'MSE_MF':>12} {'MSE_LF':>12} {'R²_MF':>8} {'R²_LF':>8} "
          f"{'σ_amp':>10} {'MSE/σ²':>10} {'FLAG':>15}")
    print("-" * 105)
    
    mse_mf_list = []
    mse_lf_list = []
    r2_mf_list = []
    g0s = []
    
    for i in range(n_sheets):
        try:
            t_i, g0_i, w_i, s_i, st_i, sr_i = extractor(i)
            g0_val = np.unique(g0_i)[0] if hasattr(g0_i, 'unique') else np.unique(g0_i)[0]
            omega_val = np.unique(w_i)[0] if hasattr(w_i, 'unique') else np.unique(w_i)[0]
            
            s_np = s_i.values if hasattr(s_i, 'values') else np.array(s_i)
            st_np = st_i.values if hasattr(st_i, 'values') else np.array(st_i)
            sr_np = sr_i.values if hasattr(sr_i, 'values') else np.array(sr_i)
            w_np = w_i.values if hasattr(w_i, 'values') else np.array(w_i)
            g0_np = g0_i.values if hasattr(g0_i, 'values') else np.array(g0_i)
            
            X_test = np.column_stack([st_np, sr_np, w_np, g0_np])
            y_test = np.column_stack([s_np])
            
            X_test_norm, y_test_norm = normalize_data(X_test, y_test, scaler_X, scaler_y)
            
            y_LF_pred = model_LF(X_test_norm)
            X_MF = np.column_stack((X_test_norm, y_LF_pred))
            y_MF_pred = model_HF_nl(X_MF) + model_HF_l(X_MF)
            
            y_LF_denorm = scaler_y.inverse_transform(y_LF_pred.numpy())
            y_MF_denorm = scaler_y.inverse_transform(y_MF_pred.numpy())
            
            mse_mf = np.mean((s_np.flatten() - y_MF_denorm.flatten()) ** 2)
            mse_lf = np.mean((s_np.flatten() - y_LF_denorm.flatten()) ** 2)
            
            s_amp = (np.max(s_np) - np.min(s_np)) / 2.0
            mse_normalized = mse_mf / (s_amp**2 + 1e-15)
            
            r2_mf = r2_score(s_np.flatten(), y_MF_denorm.flatten())
            r2_lf = r2_score(s_np.flatten(), y_LF_denorm.flatten())
            
            flag = ""
            if r2_mf < 0.8:
                flag += " LOW_R²_MF"
            if mse_normalized > 0.1:
                flag += " HIGH_NORM_MSE"
            if r2_lf < 0:
                flag += " LF_WORSE_MEAN"
            
            mse_mf_list.append(mse_mf)
            mse_lf_list.append(mse_lf)
            r2_mf_list.append(r2_mf)
            g0s.append(g0_val)
            
            print(f"{i:>5} {g0_val:>10.6f} {mse_mf:>12.6f} {mse_lf:>12.6f} {r2_mf:>8.4f} {r2_lf:>8.4f} "
                  f"{s_amp:>10.4f} {mse_normalized:>10.4f} {flag}")
        except Exception as e:
            print(f"{i:>5} ERROR: {e}")
            mse_mf_list.append(float('nan'))
            mse_lf_list.append(float('nan'))
            r2_mf_list.append(float('nan'))
            g0s.append(float('nan'))
    
    # ── Plot R² vs γ₀ ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ax = axes[0]
    ax.semilogx(g0s, r2_mf_list, 'ro-', label='R² (MFNN)', markersize=8)
    ax.semilogx(g0s, [r2_score(
        (lambda s: s.values if hasattr(s, 'values') else np.array(s))(extractor(i)[3]).flatten(),
        scaler_y.inverse_transform(model_LF(normalize_data(
            np.column_stack([(lambda x: x.values if hasattr(x, 'values') else np.array(x))(extractor(i)[4]),
                           (lambda x: x.values if hasattr(x, 'values') else np.array(x))(extractor(i)[5]),
                           (lambda x: x.values if hasattr(x, 'values') else np.array(x))(extractor(i)[2]),
                           (lambda x: x.values if hasattr(x, 'values') else np.array(x))(extractor(i)[1])]),
            np.column_stack([(lambda x: x.values if hasattr(x, 'values') else np.array(x))(extractor(i)[3])]),
            scaler_X, scaler_y)[0]).numpy()).flatten()
    ) for i in range(n_sheets)], 'b--', label='R² (LF only)', markersize=5, alpha=0.7)
    ax.axhline(0.9, color='green', linestyle=':', alpha=0.5)
    ax.axhline(0.0, color='red', linestyle=':', alpha=0.5)
    ax.set_xlabel('γ₀'); ax.set_ylabel('R²')
    ax.set_title('DIAGNOSTIC 6: R² vs γ₀ (per sheet)')
    ax.legend(); ax.grid(True, alpha=0.3)
    
    ax = axes[1]
    ax.loglog(g0s, mse_mf_list, 'ro-', label='MSE (MFNN)', markersize=8)
    ax.loglog(g0s, mse_lf_list, 'b--', label='MSE (LF only)', markersize=5, alpha=0.7)
    ax.set_xlabel('γ₀'); ax.set_ylabel('MSE (original scale)')
    ax.set_title('DIAGNOSTIC 6: MSE vs γ₀ (per sheet)')
    ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('diag6_per_sheet_loss.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return mse_mf_list, r2_mf_list


# ═══════════════════════════════════════════════════════════════════
# DIAGNOSTIC 7 — pos_finder validation
# ═══════════════════════════════════════════════════════════════════
def diagnostic_7_pos_finder(data, pos_finder):
    """
    For 5+ sheets, plot the raw strain signal and mark the detected
    zero-crossing index. Visual check for correctness.
    """
    n_sheets = len(data)
    check_indices = list(range(min(8, n_sheets)))  # Check first 8 sheets
    
    print("\n" + "=" * 100)
    print("DIAGNOSTIC 7: pos_finder Validation")
    print("=" * 100)
    
    n_cols = 4
    n_rows = int(np.ceil(len(check_indices) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
    axes = axes.flatten()
    
    for plot_idx, i in enumerate(check_indices):
        ax = axes[plot_idx]
        df_sheet = data[i][1].dropna()
        strain = df_sheet['Strain'].to_numpy()
        time_raw = df_sheet['Step time'].to_numpy()
        
        idx = pos_finder(strain)
        
        ax.plot(time_raw, strain, 'k-', linewidth=0.5, alpha=0.7)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.axvline(time_raw[idx], color='red', linestyle='-', linewidth=2, alpha=0.7)
        ax.plot(time_raw[idx], strain[idx], 'ro', markersize=10, zorder=5)
        
        g0_val = df_sheet['Oscillation strain'].iloc[0]
        ax.set_title(f'Sheet {i} (γ₀={g0_val:.4f})\nidx={idx}, t={time_raw[idx]:.3f}, γ={strain[idx]:.5f}',
                     fontsize=8)
        ax.tick_params(labelsize=7)
        
        # Print info
        print(f"  Sheet {i}: idx={idx}, t₀={time_raw[idx]:.4f}, γ(t₀)={strain[idx]:.6f}, "
              f"γ₀={g0_val:.6f}")
        if abs(strain[idx]) > 0.1 * g0_val:
            print(f"    ⚠ WARNING: Zero-crossing strain |{strain[idx]:.6f}| > 10% of γ₀={g0_val:.6f}")
    
    for j in range(len(check_indices), len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('DIAGNOSTIC 7: pos_finder Zero-Crossing Detection', fontsize=13)
    plt.tight_layout()
    plt.savefig('diag7_pos_finder.png', dpi=150, bbox_inches='tight')
    plt.show()


# ═══════════════════════════════════════════════════════════════════
# BONUS DIAGNOSTIC — Stress scale analysis
# ═══════════════════════════════════════════════════════════════════
def diagnostic_bonus_stress_scaling(data, extractor):
    """
    Check how stress amplitude scales with γ₀.
    In linear VE regime: σ_amp ∝ γ₀
    In nonlinear regime: σ_amp may saturate or show different scaling.
    This helps diagnose whether StandardScaler is appropriate.
    """
    n_sheets = len(data)
    
    print("\n" + "=" * 100)
    print("BONUS DIAGNOSTIC: Stress Amplitude Scaling with γ₀")
    print("=" * 100)
    
    g0s = []
    s_amps = []
    s_amps_over_g0 = []
    
    for i in range(n_sheets):
        t_i, g0_i, w_i, s_i, st_i, sr_i = extractor(i)
        g0_val = np.unique(g0_i)[0] if hasattr(g0_i, 'unique') else np.unique(g0_i)[0]
        s_np = s_i.values if hasattr(s_i, 'values') else np.array(s_i)
        s_amp = (np.max(s_np) - np.min(s_np)) / 2.0
        
        g0s.append(g0_val)
        s_amps.append(s_amp)
        s_amps_over_g0.append(s_amp / g0_val)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    ax = axes[0]
    ax.loglog(g0s, s_amps, 'ko-', markersize=8)
    # Plot linear reference: σ ∝ γ₀
    g0_sort = np.sort(g0s)
    ax.loglog(g0_sort, g0_sort * (s_amps[0] / g0s[0]), 'r--', alpha=0.5, label='σ ∝ γ₀ (linear VE)')
    ax.set_xlabel('γ₀')
    ax.set_ylabel('σ_amplitude')
    ax.set_title('Stress Amplitude vs γ₀')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    ax = axes[1]
    ax.semilogx(g0s, s_amps_over_g0, 'ko-', markersize=8)
    ax.set_xlabel('γ₀')
    ax.set_ylabel('σ_amp / γ₀ ≈ |G*|')
    ax.set_title('Normalized Stress Amplitude (≈ |G*|) vs γ₀\n(Should be flat in linear regime)')
    ax.grid(True, alpha=0.3)
    
    ax = axes[2]
    # How many orders of magnitude does stress span?
    s_ratio = max(s_amps) / min(s_amps)
    g_ratio = max(g0s) / min(g0s)
    ax.bar(['γ₀ range', 'σ range'], [np.log10(g_ratio), np.log10(s_ratio)], color=['blue', 'red'])
    ax.set_ylabel('Orders of magnitude (log₁₀)')
    ax.set_title(f'Dynamic Range\nγ₀: {g_ratio:.0f}x, σ: {s_ratio:.0f}x')
    ax.grid(True, alpha=0.3)
    
    print(f"\n  γ₀ range: {min(g0s):.6f} → {max(g0s):.6f} ({g_ratio:.0f}x)")
    print(f"  σ  range: {min(s_amps):.6f} → {max(s_amps):.6f} ({s_ratio:.0f}x)")
    print(f"  σ/γ₀ range: {min(s_amps_over_g0):.2f} → {max(s_amps_over_g0):.2f}")
    
    if s_ratio > 100:
        print("\n  ⚠ CRITICAL: Stress spans >2 orders of magnitude!")
        print("  StandardScaler will be dominated by high-amplitude sheets.")
        print("  Low-amplitude data is effectively invisible to MSE loss.")
        print("  → STRONGLY recommend per-amplitude normalization or log-scaling.")
    
    plt.tight_layout()
    plt.savefig('diag_bonus_scaling.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return g0s, s_amps, s_amps_over_g0


# ═══════════════════════════════════════════════════════════════════
# MASTER RUNNER
# ═══════════════════════════════════════════════════════════════════
def run_all_diagnostics_notebook1(data, X_fit_dict, Y_fit_dict, y0_fit_dict,
                                   gpr_eta, gpr_tau, X_log, Y_log,
                                   extractor, fit_function, pos_finder):
    """
    Run all diagnostics that can be run after Notebook 1.
    Call this from a notebook cell after executing Notebook 1.
    """
    print("╔" + "═" * 78 + "╗")
    print("║  MFNN LAOS DIAGNOSTICS — NOTEBOOK 1 (LF Generation Layer)                  ║")
    print("╚" + "═" * 78 + "╝")
    
    r2_scores, amp_ratios_fit = diagnostic_1_maxwell_fit_quality(
        data, X_fit_dict, Y_fit_dict, y0_fit_dict, extractor, fit_function)
    
    eta_pred, tau_pred, eta_std, tau_std = diagnostic_2_gpr_quality(
        X_fit_dict, Y_fit_dict, gpr_eta, gpr_tau, X_log, Y_log)
    
    lf_hf_ratios = diagnostic_3_lf_hf_amplitude_ratio(
        data, extractor, X_fit_dict, Y_fit_dict, y0_fit_dict, gpr_eta, gpr_tau)
    
    antisymmetry = diagnostic_4_antisymmetry(data, extractor)
    
    diagnostic_7_pos_finder(data, pos_finder)
    
    g0s, s_amps, s_over_g0 = diagnostic_bonus_stress_scaling(data, extractor)
    
    # ── Summary ──
    print("\n" + "╔" + "═" * 78 + "╗")
    print("║  SUMMARY OF FINDINGS                                                        ║")
    print("╚" + "═" * 78 + "╝")
    
    n = len(data)
    bad_maxwell = sum(1 for r in r2_scores if r < 0.5)
    bad_gpr = sum(1 for i in range(n) if eta_std[i] > 0.5 or tau_std[i] > 0.5)
    bad_ratio = sum(1 for r in lf_hf_ratios if r < 0.5 or r > 2.0)
    bad_sym = sum(1 for r in antisymmetry if r > 0.2)
    
    print(f"\n  Sheets with poor Maxwell fit (R² < 0.5):     {bad_maxwell}/{n}")
    print(f"  Sheets with high GPR uncertainty (σ > 0.5):  {bad_gpr}/{n}")
    print(f"  Sheets with LF/HF amp ratio outside [0.5,2]: {bad_ratio}/{n}")
    print(f"  Sheets with significant asymmetry (R > 0.2): {bad_sym}/{n}")
    print(f"  Stress dynamic range:                        {max(s_amps)/min(s_amps):.0f}x")
    
    return {
        'r2_scores': r2_scores,
        'lf_hf_ratios': lf_hf_ratios,
        'antisymmetry': antisymmetry,
        'eta_pred': eta_pred, 'tau_pred': tau_pred,
        'eta_std': eta_std, 'tau_std': tau_std,
        'g0s': g0s, 's_amps': s_amps
    }


def run_all_diagnostics_notebook2(data, extractor, model_LF, model_HF_nl, model_HF_l,
                                   scaler_X, scaler_y, normalize_data,
                                   X_data_LF, X_data_HF, y_data_LF, y_data_HF):
    """
    Run diagnostics that require Notebook 2's trained models.
    Call this from a notebook cell after training in Notebook 2.
    """
    print("╔" + "═" * 78 + "╗")
    print("║  MFNN LAOS DIAGNOSTICS — NOTEBOOK 2 (Post-Training)                         ║")
    print("╚" + "═" * 78 + "╝")
    
    diagnostic_5_input_distributions(X_data_LF, X_data_HF, y_data_LF, y_data_HF,
                                      scaler_X, scaler_y)
    
    mse_list, r2_list = diagnostic_6_per_sheet_loss(
        data, extractor, model_LF, model_HF_nl, model_HF_l,
        scaler_X, scaler_y, normalize_data)
    
    return {'mse_per_sheet': mse_list, 'r2_per_sheet': r2_list}

In [ ]:
# run_all_diagnostics_notebook2(data, extractor, model_LF, model_HF_nl, model_HF_l,
#                                    scaler_X, scaler_y, normalize_data,
#                                    X_data_LF, X_data_HF, y_data_LF, y_data_HF)